In [ ]:
"""
linearity calculation.
"""

In [1]:
# ============================================================
# 0. Imports & Font Configuration
# ------------------------------------------------------------
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
     "pandas==2.2.2", "geopandas==1.0.1", "numpy==1.26.4", "matplotlib==3.10.0",
     "seaborn==0.13.2", "scikit-learn==1.6.1", "scipy==1.15.2"])

import math
import os
import geopandas as gpd

import pandas as pd
import numpy as np
import seaborn as sns
import time
import matplotlib.pyplot as plt
import matplotlib.image as img
import matplotlib.gridspec as gridspec
from matplotlib import font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.ticker import FuncFormatter

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.optimize import brentq
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
os.chdir(os.getcwd())

# Load custom fonts
Tnr_path   = os.path.join(os.getcwd(), 'Times New Roman.ttf')
italic_path= os.path.join(os.getcwd(), 'Roman Italic.ttf')
for path in (Tnr_path, italic_path):
    fm.fontManager.addfont(path)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': [fm.FontProperties(fname=Tnr_path).get_name()],
    'mathtext.fontset': 'custom',
    'mathtext.rm': fm.FontProperties(fname=Tnr_path).get_name(),
    'mathtext.it': fm.FontProperties(fname=italic_path).get_name(),
})

In [7]:
# ============================================================
# 1. Load Aggregated Link & Node Layers
# ------------------------------------------------------------
cities_links = [f for f in os.listdir('data/groupedlinks/') if f.endswith('_link_betweenness.gpkg')]
part_cities_links = [gpd.read_file(os.path.join('data/groupedlinks', f)) for f in cities_links]
all_cities_links = gpd.GeoDataFrame(pd.concat(part_cities_links, ignore_index=True))

cities_nodes = [f for f in os.listdir('data/groupednodes/') if f.endswith('_node_centrality.gpkg')]
part_cities_nodes = [gpd.read_file(os.path.join('data/groupednodes', f)) for f in cities_nodes]
all_cities_nodes = gpd.GeoDataFrame(pd.concat(part_cities_nodes, ignore_index=True))
unique_sgg_nm = all_cities_links['SIGUNGU_NM'].dropna().unique()

In [8]:
# ============================================================
# 2. Helper Functions: North Arrow & Scale Bar
# ------------------------------------------------------------
def add_north_arrow(ax, x, y, arrow_file, zoom=0.05):
    im = img.imread(arrow_file)
    ax.add_artist(AnnotationBbox(OffsetImage(im, zoom=zoom), (x, y), xycoords='axes fraction', frameon=False))

def add_scale_bar(ax, length, location=(0.1,0.02), linewidth=3, color='black'):
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    sb_x = xlim[0] + (xlim[1]-xlim[0])*location[0]
    sb_y = ylim[0] + (ylim[1]-ylim[0])*location[1]
    ax.plot([sb_x, sb_x+length], [sb_y, sb_y], color=color, linewidth=linewidth)
    ax.text(sb_x+length/2, sb_y, f'{round(length/1000):,} km', va='bottom', ha='center', fontsize=10)

In [14]:
# ============================================================
# 3. Metric 1: MLI (Morphological Linearity Index)
# ------------------------------------------------------------
scaler = MinMaxScaler()
pca = PCA(n_components=1)
if not os.path.exists('data/plot/map_MLI/'):
    os.makedirs('data/plot/map_MLI/')
if not os.path.exists('data/plot/map_betweenness/'):
    os.makedirs('data/plot/map_betweenness/')

midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')
processing_city = 0
for city in unique_sgg_nm:
    processing_city += 1
    start_time = time.time()
    city_boundary = midsize_cities[midsize_cities['SIGUNGU_NM'] == city].copy()
    city_link = all_cities_links[all_cities_links['SIGUNGU_NM'] == city].copy()
    city_node = all_cities_nodes[all_cities_nodes['SIGUNGU_NM'] == city].copy()
    city_link['betweenness_div'] = city_link['betweenness']/(len(city_node)*(len(city_node)-1))
    betweenness = 'betweenness_div'
    fig, axs = plt.subplots(1, 4, figsize=(24, 7))
    city_EN = city_boundary['SIGUNGU_NM_EN'].iat[0]
    # Panel 1: centered link midpoints
    city_link['centered_x1'] = city_link.geometry.apply(lambda g: g.coords[0][0] - ((g.coords[0][0]+g.coords[-1][0])/2))
    city_link['centered_y1'] = city_link.geometry.apply(lambda g: g.coords[0][1] - ((g.coords[0][1]+g.coords[-1][1])/2))
    city_link['centered_x2'] = city_link.geometry.apply(lambda g: g.coords[-1][0] - ((g.coords[0][0]+g.coords[-1][0])/2))
    city_link['centered_y2'] = city_link.geometry.apply(lambda g: g.coords[-1][1] - ((g.coords[0][1]+g.coords[-1][1])/2))
    axs[0].plot(city_link[['centered_x1','centered_x2']].values.T,
                city_link[['centered_y1','centered_y2']].values.T,
                color='black', alpha=0.5)
    axs[0].set_title('Set Midpoint of Links\nat The Origin', fontsize=35)
    # Ensure aspect equal
    xlim, ylim = axs[0].get_xlim(), axs[0].get_ylim()
    xr, yr = xlim[1]-xlim[0], ylim[1]-ylim[0]
    if xr>yr:
        cy = (ylim[0]+ylim[1])/2; axs[0].set_ylim((cy-xr/2, cy+xr/2))
    else:
        cx = (xlim[0]+xlim[1])/2; axs[0].set_xlim((cx-yr/2, cx+yr/2))
    # Panel 2: link betweenness
    city_link['bw_scaled'] = scaler.fit_transform(city_link[[betweenness]])*5
    city_link.fillna({'bw_scaled': 0}, inplace=True)
    
    city_link.plot(ax=axs[1], color='black', linewidth=0.1)
    city_link.plot(column='bw_scaled', cmap='Reds', linewidth=city_link['bw_scaled']+0.1,
                   ax=axs[1], legend=False, alpha=0.9, zorder=1)
    city_boundary.plot(ax=axs[1], facecolor='none', edgecolor='black', linewidth=1, zorder=2)
    axs[1].legend(handles=[Line2D([0],[0],color='black',lw=0.3,label='min'),
                   Line2D([0],[0],color='darkred',lw=5,label='max')],
                   loc='center right', fontsize=20)
    axs[1].set_aspect('equal'); axs[1].axis('off'); axs[1].set_title('Calculate\nEdge Betweenness Centrality', fontsize=35)
    add_north_arrow(axs[1],0.9,0.9,'north_arrow.png',zoom=0.2); add_scale_bar(axs[1],5000,(0.9,0))
    # Panel 3: scaled line weights
    for ax_i in range(2,3+1): pass
    city_link['sxi'] = (city_link['centered_x1'] - (city_link['centered_x1']+city_link['centered_x2'])/2)*city_link[betweenness]
    city_link['syi'] = (city_link['centered_y1'] - (city_link['centered_y1']+city_link['centered_y2'])/2)*city_link[betweenness]
    city_link['sxf'] = (city_link['centered_x2'] - (city_link['centered_x1']+city_link['centered_x2'])/2)*city_link[betweenness]
    city_link['syf'] = (city_link['centered_y2'] - (city_link['centered_y1']+city_link['centered_y2'])/2)*city_link[betweenness]
    axs[2].plot(city_link[['sxi','sxf']].values.T, city_link[['syi','syf']].values.T, color='black', alpha=0.5)
    axs[2].set_title('Weight Betweenness\nCentrality', fontsize=35)
    # equal aspect
    xlim,ylim=axs[2].get_xlim(),axs[2].get_ylim();xr,yr=xlim[1]-xlim[0],ylim[1]-ylim[0]
    if xr>yr: cy=(ylim[0]+ylim[1])/2; axs[2].set_ylim((cy-xr/2,cy+xr/2))
    else: cx=(xlim[0]+xlim[1])/2; axs[2].set_xlim((cx-yr/2,cx+yr/2))
    # Panel 4: PCA end points
    line_df=pd.DataFrame({'x':city_link['sxi'],'y':city_link['syi']}).dropna()
    pca.fit(line_df); ev=pca.explained_variance_ratio_[0]
    midsize_cities.loc[midsize_cities['SIGUNGU_NM']==city,'MLI']=ev
    pc1=pca.components_[0]
    axs[3].scatter(city_link['sxi'],city_link['syi'],color='black',alpha=0.5)
    axs[3].plot([pc1[0]*-1000,pc1[0]*1000],[pc1[1]*-1000,pc1[1]*1000],color='red',lw=2,label='PC1')
    axs[3].legend(fontsize=12)
    axs[3].set_title('Extract $e_1$ of \n End Points',fontsize=35)
    axs[3].text(0.25,0.05,f'$MLI$: {ev*100:.0f}%',fontsize=30,transform=axs[3].transAxes,ha='center',va='bottom')
    axs[3].set_xlim(axs[2].get_xlim()); axs[3].set_ylim(axs[2].get_ylim())
    plt.tight_layout(); plt.savefig(f'data/plot/map_MLI/MLI_process_{city}.png',dpi=100,bbox_inches='tight')
    if city=='김포시': plt.savefig('data/plot/paper/Figure 2',dpi=1000,bbox_inches='tight')
    plt.close()
    
    # single map export
    fig,ax=plt.subplots(figsize=(8,8))
    city_link.plot(ax=ax,color='black',linewidth=0.1,zorder=0)
    city_link.plot(ax=ax,column='bw_scaled',cmap='Reds',linewidth=city_link['bw_scaled']+0.1,alpha=1,zorder=1)
    city_boundary.plot(ax=ax,facecolor='none',edgecolor='black',linewidth=1,zorder=2)
    ax.set_aspect('equal'); ax.axis('off')
    add_north_arrow(ax,0.95,0.95,'north_arrow.png',zoom=0.2); add_scale_bar(ax,5000,(0.9,0))
    plt.savefig(f'data/plot/map_betweenness/betweenness_{city}.png',dpi=100,bbox_inches='tight'); plt.close()
    plt.close()
    
    print(f'{city} done: {int(time.time()-start_time)}s {processing_city}/{len(unique_sgg_nm)}')

midsize_cities.to_file('data/midsize_cities_index.gpkg',driver='GPKG')

충주시 done: 5s 1/60
경주시 done: 5s 2/60
영주시 done: 0s 3/60
광명시 done: 1s 4/60
상주시 done: 3s 5/60
공주시 done: 3s 6/60
김포시 done: 10s 7/60
부천시 done: 2s 8/60
원주시 done: 5s 9/60
동해시 done: 1s 10/60
논산시 done: 1s 11/60
삼척시 done: 2s 12/60
의왕시 done: 1s 13/60
김해시 done: 6s 14/60
진주시 done: 3s 15/60
제천시 done: 2s 16/60
거제시 done: 4s 17/60
김천시 done: 2s 18/60
평택시 done: 5s 19/60
시흥시 done: 1s 20/60
동두천시 done: 3s 21/60
남원시 done: 2s 22/60
익산시 done: 4s 23/60
광주시 done: 2s 24/60
파주시 done: 4s 25/60
순천시 done: 5s 26/60
과천시 done: 0s 27/60
서산시 done: 2s 28/60
경산시 done: 2s 29/60
춘천시 done: 0s 30/60
여주시 done: 3s 31/60
영천시 done: 2s 32/60
당진시 done: 5s 33/60
사천시 done: 2s 34/60
화성시 done: 7s 35/60
오산시 done: 1s 36/60
나주시 done: 3s 37/60
보령시 done: 2s 38/60
강릉시 done: 1s 39/60
의정부시 done: 5s 40/60
김제시 done: 2s 41/60
남양주시 done: 4s 42/60
포천시 done: 2s 43/60
아산시 done: 4s 44/60
문경시 done: 2s 45/60
정읍시 done: 3s 46/60
군포시 done: 1s 47/60
구리시 done: 1s 48/60
서귀포시 done: 2s 49/60
이천시 done: 7s 50/60
군산시 done: 4s 51/60
제주시 done: 6s 52/60
구미시 done: 4s 53/

In [5]:
# ============================================================
# 4. Metric 2: SLI (Structural Linearity Index)
# ------------------------------------------------------------
unique_sgg_nm = all_cities_links['SIGUNGU_NM'].dropna().unique()
midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')

cmap_ryw = LinearSegmentedColormap.from_list('RedYellowWhite',['white','yellow','red'],256)
scaler_sli = MinMaxScaler()

midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')
processing_city = 0
for city in unique_sgg_nm:
    processing_city += 1
    start_time = time.time()
    city_boundary=midsize_cities[midsize_cities['SIGUNGU_NM']==city].copy()
    city_link=all_cities_links[all_cities_links['SIGUNGU_NM']==city].copy()
    city_node=all_cities_nodes[all_cities_nodes['SIGUNGU_NM']==city].copy()
    city_node['Local_SLI']=city_node['spi']/city_node['expectedpairs']
    vmin,vmax=city_node['Local_SLI'].quantile([0.05,0.95])
    fig,ax=plt.subplots(figsize=(8,8))
    city_link.plot(ax=ax,color='black',linewidth=0.1,alpha=0.5)
    city_boundary.plot(ax=ax,facecolor='none',edgecolor='black',linewidth=2,zorder=2)
    city_node.plot(ax=ax,column='Local_SLI',cmap=cmap_ryw,linewidth=0.1,edgecolor='black',
                   vmin=vmin,vmax=vmax,legend=False)
    ax.set_aspect('equal'); ax.axis('off')
    add_north_arrow(ax,0.9,0.9,'north_arrow.png',zoom=0.2); add_scale_bar(ax,5000,(0.9,0))
    plt.savefig(f'data/plot/map_SLI/SLI_{city}.png',dpi=100,bbox_inches='tight'); plt.close()
    midsize_cities.loc[midsize_cities['SIGUNGU_NM']==city,'SLI']= city_node['spi'].sum()*6/((len(city_node)+1)*(len(city_node)+2)*len(city_node)*(len(city_node)-1))
    midsize_cities.loc[midsize_cities['SIGUNGU_NM']==city,'nodes']=len(city_node)
    
    print(f'{city} done: {int(time.time()-start_time)}s {processing_city}/{len(unique_sgg_nm)}')

midsize_cities.to_file('data/midsize_cities_index.gpkg',driver='GPKG')

충주시 done: 0s 1/60
경주시 done: 0s 2/60
영주시 done: 0s 3/60
광명시 done: 0s 4/60
상주시 done: 0s 5/60
공주시 done: 0s 6/60
김포시 done: 11s 7/60
부천시 done: 0s 8/60
원주시 done: 0s 9/60
동해시 done: 0s 10/60
논산시 done: 0s 11/60
삼척시 done: 0s 12/60
의왕시 done: 0s 13/60
김해시 done: 0s 14/60
진주시 done: 0s 15/60
제천시 done: 0s 16/60
거제시 done: 0s 17/60
김천시 done: 0s 18/60
평택시 done: 0s 19/60
시흥시 done: 0s 20/60
동두천시 done: 0s 21/60
남원시 done: 0s 22/60
익산시 done: 0s 23/60
광주시 done: 0s 24/60
파주시 done: 0s 25/60
순천시 done: 0s 26/60
과천시 done: 0s 27/60
서산시 done: 0s 28/60
경산시 done: 0s 29/60
춘천시 done: 0s 30/60
여주시 done: 0s 31/60
영천시 done: 0s 32/60
당진시 done: 0s 33/60
사천시 done: 0s 34/60
화성시 done: 0s 35/60
오산시 done: 0s 36/60
나주시 done: 0s 37/60
보령시 done: 0s 38/60
강릉시 done: 0s 39/60
의정부시 done: 0s 40/60
김제시 done: 0s 41/60
남양주시 done: 0s 42/60
포천시 done: 0s 43/60
아산시 done: 0s 44/60
문경시 done: 0s 45/60
정읍시 done: 0s 46/60
군포시 done: 0s 47/60
구리시 done: 0s 48/60
서귀포시 done: 0s 49/60
이천시 done: 0s 50/60
군산시 done: 0s 51/60
제주시 done: 0s 52/60
구미시 done: 0s 53/

In [9]:
# ============================================================
# 5. Metric 3: SKI (Stark K-index) Calculation & Visualization
# ------------------------------------------------------------

midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')

def stark_index(SLI,n):
    def eq(k): return (((7*k-6)*n**3+(6*k**2-4*k)*n**2-k**3*n-2*k**3)/(k**3*(n+1)*(n+2)*(n-1))) - SLI
    try: return brentq(eq,1,200)
    except: return np.nan

def stark_index_simple(SLI):
    def eq(k): return ((7*k-6)/k**3) - SLI
    try: return brentq(eq,1,200)
    except: return np.nan

midsize_cities['SKI']=midsize_cities.apply(lambda r: stark_index(r['SLI'],r['nodes']),axis=1)
midsize_cities['SKI_simple']=midsize_cities.apply(lambda r: stark_index_simple(r['SLI']),axis=1)

minv,maxv=min(midsize_cities['SKI']),min(midsize_cities['SKI_simple'])
fig,ax=plt.subplots();
sns.scatterplot(data=midsize_cities,x='SKI',y='SKI_simple',color='darkgray',edgecolor='black',ax=ax)
ax.set_xlabel('$SKI$', fontsize=20); ax.set_ylabel('asymptotic ($N = ∞$ ) $SKI$', fontsize=20)
x_vals = np.linspace(*ax.get_xlim(), 100)
ax.plot(x_vals, x_vals, 'r--', lw=1, label='y = x',linestyle='--', zorder = 2)
ax.legend(fontsize=15, loc='upper left')
plt.savefig('data/plot/paper/Figure 4.png',dpi=1000,bbox_inches='tight'); plt.close()


midsize_cities['SKI_diff'] = midsize_cities['SKI'] - midsize_cities['SKI_simple']
# show max SKI_diff with city English name and node count
if midsize_cities['SKI_diff'].dropna().empty:
    print("No SKI_diff values available")
else:
    idx = midsize_cities['SKI_diff'].idxmax()
    row = midsize_cities.loc[idx]
    en_name = row.get('SIGUNGU_NM_EN', row.get('SIGUNGU_NM', 'Unknown'))
    nodes = int(row['nodes']) if pd.notna(row.get('nodes')) else row.get('nodes')
    print(f"Max difference: {midsize_cities['SKI_diff'].max():.6f} — {en_name} (nodes: {nodes})")
print(f'Mean difference: {midsize_cities["SKI_diff"].mean()}')
print(f'Max variance: {midsize_cities["SKI"].max() - midsize_cities["SKI"].min()}')

midsize_cities.to_file('data/midsize_cities_index.gpkg', driver='GPKG')

/tmp/ipykernel_173917/2806791974.py:25: UserWarning: linestyle is redundantly defined by the 'linestyle' keyword argument and the fmt string "r--" (-> linestyle='--'). The keyword argument will take precedence.
  ax.plot(x_vals, x_vals, 'r--', lw=1, label='y = x',linestyle='--', zorder = 2)


Max difference: 1.128939 — Bucheon (nodes: 2562)
Mean difference: 0.5322640814214386
Max variance: 71.71443811617007


In [10]:
# ============================================================
# 6. Paper-ready Summary Figures (MLI & SLI Grids)
# ------------------------------------------------------------

# Define the range for n and SLI
n_values = np.linspace(midsize_cities['nodes'].min(), midsize_cities['nodes'].max(), 100)
pi_linearity_values = np.linspace(midsize_cities['SLI'].min(), midsize_cities['SLI'].max(), 100)

# Create a meshgrid for n and SLI
n_grid, SLI_grid = np.meshgrid(n_values, pi_linearity_values)

# Calculate petal_index for each combination of N and SLI
SKI_grid = np.vectorize(stark_index)(SLI_grid, n_grid)

# Plot the results side by side with adjusted aspect ratio
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 2])  # Adjust width ratios for better visualization

# First plot
ax1 = fig.add_subplot(gs[0], projection='3d')
ax1.plot_surface(n_grid, SLI_grid, SKI_grid, edgecolor='none')
ax1.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:,.0f}'))
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:,.2f}'))
ax1.set_xlabel(r'$N$', fontsize=20, labelpad=15)
ax1.set_ylabel(r'$SLI$', fontsize=20, labelpad=10)
ax1.set_zlabel(r'$SKI$', fontsize=20, labelpad=10)

# Second plot
ax2 = fig.add_subplot(gs[1], projection='3d')
ax2.plot_surface(n_grid, SLI_grid, SKI_grid, edgecolor='none')
ax2.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:,.0f}'))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:,.2f}'))
ax2.view_init(elev=0, azim=180)
ax2.set_xlabel(r'$N$', fontsize=20, labelpad=15)
ax2.set_ylabel(r'$SLI$', fontsize=20, labelpad=10)
ax2.set_zlabel(r'$SKI$', fontsize=20, labelpad=10)

plt.tight_layout()
plt.savefig('data/plot/paper/Figure 3.png', dpi=1000, bbox_inches='tight')
plt.close()

In [11]:
# ============================================================
# 7. Single‑City local SLI & SKI Visualization
# ------------------------------------------------------------

# Configure colormap for Local_SLI visualization
cmap = plt.get_cmap('autumn_r', 256)  # autumn_r 컬러맵 사용

# Select the target city and extract its geometries
city = '김포시'
city_boundary = midsize_cities[midsize_cities['SIGUNGU_NM'] == city].copy()
city_link     = all_cities_links[all_cities_links['SIGUNGU_NM'] == city].copy()
city_node     = all_cities_nodes[all_cities_nodes['SIGUNGU_NM'] == city].copy()

# Compute and scale Local_SLI for each node
city_node['Local_SLI'] = city_node['spi'] / city_node['expectedpairs']

# Retrieve the integer SKI (number of spokes)
k = int(round(city_boundary['SKI'].values[0]))

# Retrieve the English name of the city
city_EN_name = city_boundary['SIGUNGU_NM_EN'].values[0]

# Create the figure and grid layout
fig = plt.figure(figsize=(18, 8))
fig.suptitle(f'                       {city_EN_name}', fontsize=35)
gs  = gridspec.GridSpec(1, 2, width_ratios=[2, 1])  # 왼쪽을 3, 오른쪽(극좌표)을 1 비율로

# Prepare the map axis
SLI_val = round(city_boundary['SLI'].values[0] * 100, 2)
ax_map = fig.add_subplot(gs[0])
ax_map.set_title(fr'$SLI = {SLI_val}\times 10^{{-2}}$', fontsize=25, fontname='Times New Roman')

# Plot road network and city boundary
city_link.plot(ax=ax_map, color='black', linewidth=1, alpha=0.7)
city_boundary.plot(ax=ax_map, facecolor='none', edgecolor='black', linewidth=2, zorder=2)

values = city_node['Local_SLI'].values
vmin = city_node['Local_SLI'].quantile(0.05)
vmax = city_node['Local_SLI'].quantile(0.95)

# Plot nodes colored by Local_SLI
city_node.plot(
    column='Local_SLI',
    cmap=cmap,
    linewidth=0.1,
    ax=ax_map,
    legend=False,
    vmin=vmin,
    vmax=vmax,
    edgecolor='black'
)


# Add an inset legend (scatter showing value distribution)
ax_dist = inset_axes(
    ax_map,
    width="8%", 
    height="80%",
    loc='right',
    borderpad=-5
)

ax_dist.set_facecolor('white')
ax_dist.scatter(
    np.random.default_rng(42).uniform(0, 1, size=len(values)),
    values,
    s=8,
    alpha=1,
    vmin=vmin,
    vmax=vmax,
    c=values,
    cmap=cmap
)
ax_dist.set_title('Local $SLI$', fontsize=20, pad=10)

# Configure y-axis ticks and grid for the inset
y_ticks = np.arange(vmin, vmax + 0.5, 0.5)
ax_dist.set_ylim(vmin, vmax)
ax_dist.yaxis.tick_right()
ax_dist.set_xticks([])
ax_dist.tick_params(labelsize=20)
ax_dist.grid(axis='y', linestyle='--', color='black', alpha=0.8, linewidth=1.5)

# Add north arrow and scale bar
add_north_arrow(ax_map, x=0.85, y=0.9, arrow_file='north_arrow.png', zoom=0.2)
add_scale_bar(ax_map, location=(0.05, 0.05), length=5000)

ax_map.set_aspect('equal')
ax_map.axis('off')


# Prepare the polar axis for SKI spokes
ax_polar = fig.add_subplot(1, 2, 2, polar=True)
ax_polar.spines['polar'].set_visible(False)
ax_polar.set_theta_zero_location('N')
ax_polar.set_theta_direction(-1)
ax_polar.set_ylim(0, 1)
ax_polar.grid(False)
ax_polar.set_xticks([])
ax_polar.set_yticks([])

# Draw each spoke line
theta_vals = np.linspace(0, 2 * np.pi, k, endpoint=False)
for theta in theta_vals:
    ax_polar.plot([theta, theta], [0, 1], linewidth=2, color='black')

ax_polar.set_title(rf"$SKI$ = {k}", fontsize=25, pad=25)

# Final layout adjustment, save
plt.tight_layout()
plt.savefig(f'data/plot/paper/Figure 5.png', dpi=1000, bbox_inches='tight')
plt.close()

/tmp/ipykernel_173917/3980528520.py:110: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [12]:
# ============================================================
# 8. Summary Figure: MLI Top/Bottom 8 Cities
# ------------------------------------------------------------

# setup
n = 8                         # 상위 8개 · 하위 8개
metric_col   = 'MLI'
folder       = 'data/plot/map_betweenness'
out_dir      = 'data/plot/summary'
os.makedirs(out_dir, exist_ok=True)

# load midsize cities
cities = gpd.read_file('data/midsize_cities_index.gpkg')
cities = cities[cities['SIGUNGU_NM'] != '양산시']


# filter top/bottom n cities by MLI
top_df    = cities.nlargest(n, metric_col)
bottom_df = cities.nsmallest(n, metric_col)
city_list = list(top_df.SIGUNGU_NM) + list(bottom_df.SIGUNGU_NM)

# calculate rows and columns for subplots
cols = 4
rows = math.ceil(len(city_list) / cols)  # 16개 → 4행

# create figure and axes
fig, axes = plt.subplots(rows, cols, figsize=(8.27, 11))
axes = axes.flatten()
fig.suptitle(r'$MLI$', fontsize=25, y=0.95)

# Load and display images for each city
for ax, city_nm in zip(axes, city_list):
    rec      = cities[cities.SIGUNGU_NM == city_nm].iloc[0]
    city_en  = rec.SIGUNGU_NM_EN
    metric_v = rec[metric_col]
    
    img_path = os.path.join(folder, f'betweenness_{city_nm}.png')
    if os.path.exists(img_path):
        ax.imshow(img.imread(img_path))
    ax.axis('off')
    ax.set_title(f"{city_en} \n {metric_v:.3f}", fontsize=12, pad=6)

# Hide any unused axes
for ax in axes[len(city_list):]:
    ax.axis('off')

# Adjust layout and spacing
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.94])

# row divider
y_div = axes[cols].get_position().y0  
divider = Line2D([0.05, 0.95], [y_div-0.03, y_div-0.03],
                 transform=fig.transFigure,
                 color='black', linewidth=1)
fig.add_artist(divider)

# text labels for top/bottom sections
fig.text(0.01, 0.75, 'Top 8',    va='center', ha='left',
         rotation=90, fontsize=15)
fig.text(0.01, 0.25, 'Bottom 8', va='center', ha='left',
         rotation=90, fontsize=15)

# Add a legend for the color gradient
from matplotlib.patches import Polygon
leg_ax = fig.add_axes([0.60, 0.935, 0.25, 0.06], frameon=False)

# Create a gradient for the legend
grad = np.linspace(0, 1, 256).reshape(1, -1)

# Create an image object for the gradient
im = leg_ax.imshow(
    grad,
    cmap='Reds',
    aspect='auto',
    extent=(0, 1, 0, 1),
    origin='lower',
    clip_on=True
)

# Create a triangle patch for clipping
triangle = Polygon(
    [[1, 0], [1, 0.12], [0.4, 0.06]],
    transform=leg_ax.transAxes,
    facecolor='none'
)
leg_ax.add_patch(triangle)

# clip the image to the triangle
im.set_clip_path(triangle)

# Add colorbar-like labels
leg_ax.text(0.4, -0.1, 'Min', ha='center', va='top', transform=leg_ax.transAxes, fontsize=12)
leg_ax.text(1, -0.1, 'Max', ha='center', va='top', transform=leg_ax.transAxes, fontsize=12)
leg_ax.text(0.7, 0.4, 'Betweenness Centrality', ha='center', va='top', transform=leg_ax.transAxes, fontsize=12)

leg_ax.axis('off')

# Save the figure
out_path = os.path.join(out_dir, 'MLI.png')
fig.savefig(out_path, dpi=100, bbox_inches='tight')
fig.savefig('data/plot/paper/Figure 6.png', dpi=300, bbox_inches='tight')
plt.close()

In [13]:
# ============================================================
# 9. Summary Figure: SLI Top/Bottom 8 Cities
# ------------------------------------------------------------


# setup
cmap_ryw = LinearSegmentedColormap.from_list(
    'RedYellowWhite',
    ['white', 'yellow', 'red'],
    N=256
)
cmap = cmap_ryw

n           = 8                           # 상위 8개 · 하위 8개
metric_col  = 'SLI'
map_folder  = 'data/plot/map_SLI'         # SLI 맵이 저장된 폴더
out_folder  = 'data/plot/summary'
os.makedirs(out_folder, exist_ok=True)

# load midsize cities
cities = gpd.read_file('data/midsize_cities_index.gpkg')
cities = cities[cities['SIGUNGU_NM'] != '양산시']


# filter top/bottom n cities by SLI
top_df    = cities.nlargest(n, metric_col)
bottom_df = cities.nsmallest(n, metric_col)
city_list = list(top_df.SIGUNGU_NM) + list(bottom_df.SIGUNGU_NM)

# calculate rows and columns for subplots
cols = int(n/2)
rows = math.ceil(len(city_list) / cols)   # 총 16개 → 4행

# create figure and axes
fig, axes = plt.subplots(rows, cols, figsize=(8.27, 11))
axes = axes.flatten()

# Set the main title for the figure
fig.suptitle(r"$SLI\ (SKI)$", fontsize=25, y=0.95)

# Load and display images for each city
for ax, city_nm in zip(axes, city_list):
    rec     = cities[cities.SIGUNGU_NM == city_nm].iloc[0]
    city_en = rec.SIGUNGU_NM_EN
    sli_v   = rec['SLI']
    ski_v   = rec['SKI']
    
    # Load the SLI image for the city
    img_path = os.path.join(map_folder, f"SLI_{city_nm}.png")
    if os.path.exists(img_path):
        ax.imshow(img.imread(img_path))
    ax.axis('off')
    
    # Set the title for each subplot
    ax.set_title(
        f"{city_en}\n{sli_v:.4f}({ski_v:.0f})",
        fontsize=12, pad=6
    )

# Hide any unused axes
for ax in axes[len(city_list):]:
    ax.axis('off')

# Adjust layout and spacing
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.94])

# Add a horizontal line to separate the top and bottom sections
y_div = axes[cols].get_position().y0
divider = Line2D([0.05, 0.95], [y_div-0.012, y_div-0.012],
                 transform=fig.transFigure,
                 color='black', linewidth=1)
fig.add_artist(divider)

# Add text labels for the top and bottom sections
fig.text(0.01, 0.75, 'Top 8',    va='center', ha='left',
         rotation=90, fontsize=14)
fig.text(0.01, 0.25, 'Bottom 8', va='center', ha='left',
         rotation=90, fontsize=14)

# Add a legend for the color gradient

vals = np.random.normal(loc=0.5, scale=0.25, size=200)
vals = np.clip(vals, 0, 1)

# Add a legend for the color gradient
ax_dist = fig.add_axes([0.65, 0.92, 0.28, 0.03], frameon=False)  # 테두리 제거

vals = np.linspace(0, 20, 200)

ax_dist.scatter(
    vals,
    np.full(len(vals), 0.5),
    s=30,
    c=vals,
    cmap=cmap_ryw,
    vmin=0, vmax=1,
    edgecolors='black',
    linewidth=0.5
)
ax_dist.set_xlim(-0.05, 1.05)
ax_dist.set_ylim(0, 1)
ax_dist.set_yticks([])
ax_dist.set_xticks([])
ax_dist.set_axis_off()

ax_dist.text(0,   -0.2, 'Min',        ha='center', va='top', transform=ax_dist.transAxes, fontsize=12)
ax_dist.text(1,   -0.2, 'Max',        ha='center', va='top', transform=ax_dist.transAxes, fontsize=12)
ax_dist.text(0.5,  1.5, 'Local $SLI$', ha='center', va='top', transform=ax_dist.transAxes, fontsize=12)

# Save the figure
out_path = os.path.join(out_folder, 'SLI.png')
fig.savefig(out_path, dpi=100, bbox_inches='tight')
fig.savefig('data/plot/paper/Figure 7.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# ============================================================
# **. Correlation Analysis of SLI, MLI, and Network Metrics
# ------------------------------------------------------------


midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')
for city in unique_sgg_nm:
    start_time = time.time()
    

    city_link     = all_cities_links[all_cities_links['SIGUNGU_NM'] == city].copy()
    city_node     = all_cities_nodes[all_cities_nodes['SIGUNGU_NM'] == city].copy()
    city_node['Local_SLI'] = city_node['spi'] / city_node['expectedpairs']
    midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'links'] = len(city_link)
    midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'links_length'] = city_link['LENGTH'].sum()
    midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'nodebylink'] = len(city_node) / city_link['LENGTH'].sum()
    

midsize_cities['area'] = midsize_cities.geometry.area / 1e6  # Convert to square kilometers
midsize_cities['linksbyarea'] = midsize_cities['links_length'] / midsize_cities['area']
    
import scipy.stats as stats

cols = ['SLI', 'MLI', 'nodebylink', 'linksbyarea']

# Compute correlation matrix
corr_matrix = midsize_cities[cols].corr()

# Compute p-value matrix
p_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=corr_matrix.columns, index=corr_matrix.index)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        corr, p_val = stats.pearsonr(midsize_cities[cols[i]], midsize_cities[cols[j]])
        p_matrix.iloc[i, j] = p_val
        p_matrix.iloc[j, i] = p_val

print("Correlation matrix:")
print(corr_matrix)
print("\nP-value matrix:")
print(p_matrix)

midsize_cities.to_file('data/midsize_cities_index.gpkg', driver='GPKG')